<a href="https://colab.research.google.com/github/jaewoo-cho/jaewoo/blob/master/55_%EC%9D%B8%EC%BD%94%EB%8D%94_%EB%94%94%EC%BD%94%EB%8D%94_Seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sequence-to-Seqeunce
* Sequence-to-sequence(Seq2Seq)는 입력된 시퀀스로부터 다른 도메인의 시퀀스를 출력하는 모델
* 대표적인 응용분야
  - **기계번역 (machine translation)**
    - '한국어 도메인' 을 가지는 문장을 입력하면 '영어 도메인' 에 해당하는 문장을 얻을수 있다.
    - 구글 번역기, 파파고...
  - **내용 용약 (Text Summarization)** : 상대적으로 큰 원문의 핵심 내용만 간추려서 상대적으로 작은 요약문으로 변환하는 것
  - **음성 인식 (Speech to Text,STT)** : 음성을 글자로 변환하는 기술


In [ ]:
# seq2seq 는 새로운 모델이 아니라 기존 RNN(순환신경망) 2개를 조합해서 만듭니다.

## 인코더 & 디코더

* Seq2Seq는 다른 특별한 기술을 이용하는 것이 아니라, 지금까지 배운 RNN(순환신경망) 기술들을 조합해 만들며, 일반적으로 **encoder**와 **decoder**로 구성
<br>

![](https://wikidocs.net/images/page/24996/%EC%8B%9C%ED%80%80%EC%8A%A4%ED%88%AC%EC%8B%9C%ED%80%80%EC%8A%A4.PNG)


![](https://wikidocs.net/images/page/24996/seq2seq%EB%AA%A8%EB%8D%B811.PNG)

- Seq2Seq 는 '인코더' 와 '디코더' 라는 모듈로 구성됨.
- **인코더**
  - 입력문장의 모든 단어들을 순차적으로 입력 받은뒤
  - 마지막에 이 모든 단어 정보들을 압축하여 하나의 벡터를 만든다. 이를 **컨텍스트 벡터(context vector)** 라 함
  - 이 컨텍스트 벡터가 디코더에 전송된다.
- **디코더**
  - 컨텍스트 벡터 를 받아서 번역된 단어를 한개씩 순차적으로 출력.




* **Context Vector**

![](https://wikidocs.net/images/page/24996/%EC%BB%A8%ED%85%8D%EC%8A%A4%ED%8A%B8_%EB%B2%A1%ED%84%B0.PNG)

위의 그림에서는 컨텍스트 벡터를 4의 사이즈로 표현하였지만,
실제 현업에서 사용되는 seq2seq 모델에서는 보통 수백 이상의 차원을 갖고있습니다


## 인코더와 디코더의 아키텍쳐

![](https://wikidocs.net/images/page/24996/%EC%9D%B8%EC%BD%94%EB%8D%94%EB%94%94%EC%BD%94%EB%8D%94%EB%AA%A8%EB%8D%B8.PNG)

- 인코더와 디코더라 불리우는 두개의 RNN 아키텍처로 구성
  - 인코더 : 입력문장을 입력받는 RNN셀
  - 디코더 : 출력문장을 출력하는 RNN셀
  - 일반적으로 바닐라RNN 보다는 **LSTM** 이나 **GRU** 로 구성함 (성능때문에!)

- Seq2Seq 는 **'훈련과정'** 과 **'예측(테스트)과정'** 의 작동방식이 조금 다릅니다.    


### '훈련과정'의 아키텍쳐
- 인코더에는 **'입력 단어'**들이 입력 하여 컨텍스트 벡터 출력
- 디코더에게 인코더가 보낸 '컨텍스트 벡터'와 **'실제 정답'**인 &lt;sos&gt; je suis étudiant를 입력 받았을 때,
je suis étudiant &lt;eos&gt;가 나와야 된다고 **'정답'을 알려주면서 훈련**합니다.

- 즉, 훈련과정에는 위의 **3가지 데이터**가 필요하다!

- 이에 대해서는 뒤에 **교사 강요(teacher forcing)** 라는것을 설명하면서 재언급하겠습니다.


### '예측(테스트) 과정' 의 아키텍쳐
- 인코더
  - 입력문장은 단어 토큰화 되어 쪼개어 지고
  - 단어토큰 각각은 RNN셀의 각 타임스텝의 입력이 된다.
  - 인코더 RNN셀은 모든 단어를 입력받은 뒤에,
  - 인코더의 마지막 타임스텝 의 은닉상태(hidden state) 를 디코더로 넘겨진다 (이게 바로 context vector 다)
  - 이 context vector 는 디코더 RNN셀의 첫번째 은닉상태에 사용된다.


- 디코더
  - 우선 디코더는 초기 입력으로 문장의 시작을 의미하는 토큰
  &lt;sos&gt; 가 들어감
  - 디코더는 &lt;sos&gt; 가 입력되면 다음에 등장할 확률이 높은 단어를 예측.
  - 위 그림에서 첫번째 타입스텝의 디코더 RNN셀은 다음에 등장할 단어로 "je"를 예측했다.
  - 그렇게 예측한 단어 je 를 다음 타임스텝의 RNN 셀 입력으로 입력
  - .. 다음에는 "suis", 다음에는 "etudiant"..
  - 이런식으로 디코더는 **타임스텝을 거듭 반복**하면서 다음의 단어들을 예측해낸다.  언제까지?
  - 이 반복행위는 문장의 끝을 의미하는 토큰 &lt;eos&gt; 가 다음단어로 예측될때까지 반복됨.


## word embedding
- 아래 그림에서 입,출력에 쓰이는 단어토큰 부분 주목
![](https://wikidocs.net/images/page/24996/%EB%8B%A8%EC%96%B4%ED%86%A0%ED%81%B0%EB%93%A4%EC%9D%B4.PNG)

- seq2seq 에 사용되는 모든 단어들은 임베딩 벡터로 변환후 입력으로 사용됨.


![](https://wikidocs.net/images/page/24996/%EC%9E%84%EB%B2%A0%EB%94%A9%EB%B2%A1%ED%84%B0.PNG)


하나의 RNN 셀은 각각의 timestep 마다 두개의 입력을 받는다
![](https://wikidocs.net/images/page/24996/rnn%EA%B7%BC%ED%99%A9.PNG)


## softmax

![](https://wikidocs.net/images/page/24996/decodernextwordprediction.PNG)
- 각 타임스텝의 출력 단어로 나올수 있는 단어는 '다양'하다
- 이를 예측하기 위해 softmax 함수 사용됨


# 기계번역 (Machine Translation)

## 문자 레벨 기계번역
Character-Level Neural Machine Translation


In [ ]:
# 기계번역기를 훈련시키기 위한 말뭉치는 병렬 코퍼스 (parallel corpus) 필요

# 다운로드 링크 : http://www.manythings.org/anki

## 병렬 corpus 데이터

기계번역기를 훈련시키기 위해서는 훈련데이터를 병렬 코퍼스 (parrel corpus) 형태로 구성된 데이터가 필요하다.

- 출처: http://www.manythings.org/anki


  - **fra.txt http://www.manythings.org/anki/fra.txt**

- 예를 들면

|src|tar|
|---|---|
|Watch me|Regradez-moi|
|Go.|Marche.|


* 일반적인 자연어 처리의 경우, 입력 시퀀스와 출력 시퀀스의 길이가 동일함
* 그러나! Seq2Seq는 입력 시퀀스와 출력 시퀀스의 길이가 다를 수 있다고 가정
* Seq2Seq에는 다음의 데이터들이 필요하다
  - 인코더의 입력에 헤당하는 데이터
  - 디코더의 입력에 해당하는 데이터
  - 디코더의 출력과 비교할 목표 데이터 구성




## 데이터 준비

* 데이터는 영어 문장과 그에 해당하는 프랑스어 문장이 존재하는 기계 번역 데이터를 사용
* url 주소에서 데이터를 받아오고, 필요없는 열(lic)은 제거
* http://www.manythings.org/anki/fra-eng.zip



In [ ]:
base_path = r'/content/drive/MyDrive/dataset'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import tensorflow as tf
from tensorflow import keras

import random
def set_seed(seed = 42):
  tf.keras.utils.set_random_seed(seed)
  tf.config.experimental.enable_op_determinism()

set_seed(42)

In [ ]:
lines = pd.read_csv(os.path.join(base_path, 'fra.txt'), names=['src', 'tar', 'lic'],sep='\t')
lines.head()

,src,tar,lic
0,Go.,Va !,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1,Go.,Marche.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2,Go.,En route !,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
3,Go.,Bouge !,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
4,Hi.,Salut !,CC-BY 2.0 (France) Attribution: tatoeba.org #5...


In [ ]:
lines.shape

(232736, 3)

## 전처리

In [ ]:
del lines['lic'] # 불필요한 컬럼 제거

lines.head()

,src,tar
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


In [ ]:
lines.head()

,src,tar
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


*   데이터를 모두 사용할 경우 많은 시간이 소요되기 때문에, 일부 데이터만 사용
  * 게다가 Colab 에선 메모리 부족으로 학습 못하기도 함
*   목표 데이터에는 시작과 끝을 나타내는 토큰이 포함되어야 함
  *   여기서는 '\t'와 '\n'을 각각 시작과 끝을 나타내는 토큰으로 사용


In [ ]:
lines.loc[:, 'src':'tar'].head()

,src,tar
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


In [ ]:
lines = lines[0:60000]

In [ ]:
lines.sample(10)

,src,tar
12628,Please try one.,Essayez-en un.
37730,We have to be firm.,Nous devons être fermes.
39991,Give Tom some water.,Donne de l'eau à Tom.
8525,I've got eyes.,Je suis pourvu d'yeux.
8279,I'll get this.,Je vais prendre ceci.
51012,"Let's go by taxi, OK?","Allons-y en taxi, d'accord ?"
14871,Girls are crazy.,Les filles sont folles.
15127,Here's your key.,C'est ta clef.
9366,They're alive.,Ils sont en vie.
33322,I admire your work.,J'admire votre travail.


In [ ]:
lines.tar = lines.tar.apply(lambda x: '\t' + x + '\n') # <sos> ~ <eos> 추가

# 사전, 인덱스 구성

In [ ]:
# 이번 예제는 '문자레벨의 기계번역' (단어 레벨의 기계번역 아님)



*   이번 예제에서는 **글자 단위**로 예측 하기 위해서, 글자 집합을 구축해주어야 함
*   구축한 다음, 정렬해 인덱스를 부여해 '글자'에 해당하는 **사전**을 만듬
*   **사전**은 '글자'를 모델에 투입하도록 변환하거나 예측시 반환되는 인덱스들을 '글자'로 변환할 때 사용




In [ ]:
# 문자 사전 구축

src_vocab = set()
for line in lines.src: # 1줄씩 읽음
  for char in line: # 1개의 문자씩 읽음
    src_vocab.add(char)

tar_vocab = set()
for line in lines.tar: # 1줄씩 읽음
  for char in line: # 1개의 문자씩 읽음
    tar_vocab.add(char)

print(src_vocab)
print(tar_vocab)

{'g', 'a', 'J', 'D', 'y', '2', 'H', 'N', 'p', "'", '5', '3', 'r', ',', '7', '8', 's', 'v', 'ï', 'l', 'S', ':', '1', 'z', 'h', 'b', 'x', 'Z', 'k', '%', 'Y', '4', '&', ' ', 'A', '.', 'C', 'X', 'V', 'G', 'f', '?', '-', 'P', 'L', 'K', 'e', 'w', '6', 'u', 'R', '0', '!', 't', '’', '"', 'm', 'c', 'q', 'F', '€', 'M', 'Q', 'W', '/', 'd', 'é', 'o', 'B', 'T', '9', 'i', '$', 'E', 'j', 'n', 'U', 'I', 'O'}
{'g', 'â', 'a', 'J', 'D', 'y', '2', 'H', 'N', 'p', "'", '5', 'Ô', '3', 'r', ',', '7', 'À', '8', 's', 'v', 'î', 'ï', 'l', 'S', ':', 'z', '1', '\xa0', 'É', 'h', 'b', 'ë', 'ê', 'x', 'k', '%', 'Y', '4', '\n', '&', ' ', 'A', '.', 'C', 'X', '‽', 'V', 'è', 'G', 'Ê', 'Ç', 'f', 'û', '?', '-', 'ô', 'P', 'L', 'K', 'ç', 'e', 'w', '6', 'u', 'ù', 'R', '0', '!', 't', '’', '»', '"', 'm', '«', 'c', 'q', 'F', '\u202f', 'M', 'Q', 'W', 'd', 'é', 'o', 'œ', 'B', 'T', '9', 'i', '$', 'E', 'j', 'n', '\t', 'à', 'U', '‘', '\u2009', 'I', 'O'}


In [ ]:
# 사전크기
src_vocab_size = len(src_vocab) + 1
tar_vocab_size = len(tar_vocab) + 1


print('src 문장의 char 집합', src_vocab_size)
print('tar 문장의 char 집합', tar_vocab_size)

src 문장의 char 집합 80
tar 문장의 char 집합 102


In [ ]:
# 사전정렬
src_vocab = sorted(list(src_vocab))
tar_vocab = sorted(list(tar_vocab))

print(src_vocab)
print(tar_vocab)

[' ', '!', '"', '$', '%', '&', "'", ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'é', 'ï', '’', '€']
['\t', '\n', ' ', '!', '"', '$', '%', '&', "'", ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\xa0', '«', '»', 'À', 'Ç', 'É', 'Ê', 'Ô', 'à', 'â', 'ç', 'è', 'é', 'ê', 'ë', 'î', 'ï', 'ô', 'ù', 'û', 'œ', '\u2009', '‘', '’', '\u202f', '‽']


In [ ]:
# 인덱스 구성을 위해

In [ ]:
src_to_index = dict([(word, i+1) for i, word in enumerate(src_vocab)])
tar_to_index = dict([(word, i+1) for i, word in enumerate(tar_vocab)])

print(src_to_index)
print(tar_to_index)

{' ': 1, '!': 2, '"': 3, '$': 4, '%': 5, '&': 6, "'": 7, ',': 8, '-': 9, '.': 10, '/': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, ':': 22, '?': 23, 'A': 24, 'B': 25, 'C': 26, 'D': 27, 'E': 28, 'F': 29, 'G': 30, 'H': 31, 'I': 32, 'J': 33, 'K': 34, 'L': 35, 'M': 36, 'N': 37, 'O': 38, 'P': 39, 'Q': 40, 'R': 41, 'S': 42, 'T': 43, 'U': 44, 'V': 45, 'W': 46, 'X': 47, 'Y': 48, 'Z': 49, 'a': 50, 'b': 51, 'c': 52, 'd': 53, 'e': 54, 'f': 55, 'g': 56, 'h': 57, 'i': 58, 'j': 59, 'k': 60, 'l': 61, 'm': 62, 'n': 63, 'o': 64, 'p': 65, 'q': 66, 'r': 67, 's': 68, 't': 69, 'u': 70, 'v': 71, 'w': 72, 'x': 73, 'y': 74, 'z': 75, 'é': 76, 'ï': 77, '’': 78, '€': 79}
{'\t': 1, '\n': 2, ' ': 3, '!': 4, '"': 5, '$': 6, '%': 7, '&': 8, "'": 9, ',': 10, '-': 11, '.': 12, '0': 13, '1': 14, '2': 15, '3': 16, '4': 17, '5': 18, '6': 19, '7': 20, '8': 21, '9': 22, ':': 23, '?': 24, 'A': 25, 'B': 26, 'C': 27, 'D': 28, 'E': 29, 'F': 30, 'G': 31, 'H': 32, 'I': 33, 'J': 3

In [ ]:
src_vocab_size, len(src_to_index)

(80, 79)

## 인코더에 입력할 입력데이터 구성
*   인코더에 입력될 입력 데이터를 구성
*   문장의 글자 하나씩을 사전을 이용해 인덱스로 변환해 리스트에 넣음


In [ ]:
encoder_input = []
for line in lines.src:
  encoder_input.append([src_to_index[char] for char in line])

print(lines.src.values[:3])
print(encoder_input[:3])

['Go.' 'Go.' 'Go.']
[[30, 64, 10], [30, 64, 10], [30, 64, 10]]


In [ ]:
print(lines.src.values[[3, 30, 90]])
print(encoder_input[3], encoder_input[30], encoder_input[90])

['Go.' 'Help!' 'Cheers!']
[30, 64, 10] [31, 54, 61, 65, 2] [26, 57, 54, 54, 67, 68, 2]


## 디코더에 입력할 입력 데이터 구성
*   디코더에 입력될 입력 데이터를 구성
*   인코더 입력 데이터 처리와 동일하나, **목표 데이터에 해당하는 사전을 사용**해주어야 함


In [ ]:
decoder_input = []
for line in lines.tar:
  decoder_input.append([tar_to_index[char] for char in line])

print(lines.tar.values[:3])
print(decoder_input[:3])

['\tVa !\n' '\tMarche.\n' '\tEn route !\n']
[[1, 46, 50, 3, 4, 2], [1, 37, 50, 67, 52, 57, 54, 12, 2], [1, 29, 63, 3, 67, 64, 70, 69, 54, 3, 4, 2]]


## 디코더의 출력과 비교할 목표 데이터 구성
*   디코더의 출력과 비교할 목표 데이터(정답)를 구성
*   디코더의 입력 데이터를 구성할 때와 동일하나, 시작 토큰을 제외해주어야 함


In [ ]:
decoder_target = []
for line in lines.tar:
  timestep = 0
  encoded_line = []
  for char in line:
    if timestep > 0: # 시작토큰 <sos> '\t' 은 제외
      encoded_line.append(tar_to_index[char])
    timestep += 1

  decoder_target.append(encoded_line)

# 비교해보자
print(decoder_input[:5])
print(decoder_target[:5])

[[1, 46, 50, 3, 4, 2], [1, 37, 50, 67, 52, 57, 54, 12, 2], [1, 29, 63, 3, 67, 64, 70, 69, 54, 3, 4, 2], [1, 26, 64, 70, 56, 54, 3, 4, 2], [1, 43, 50, 61, 70, 69, 3, 4, 2]]
[[46, 50, 3, 4, 2], [37, 50, 67, 52, 57, 54, 12, 2], [29, 63, 3, 67, 64, 70, 69, 54, 3, 4, 2], [26, 64, 70, 56, 54, 3, 4, 2], [43, 50, 61, 70, 69, 3, 4, 2]]


## 패딩 처리


* 각각의 데이터를 동일한 길이로 맞춰줌
* 길이를 맞춰줄 때는 해당 데이터의 **최대 길이**로 맞춰줌




In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# src, tar 의 최대 길이

max_src_len = max([len(line) for line in lines.src])
max_tar_len = max([len(line) for line in lines.tar])

print('src문장 최대 길이', max_src_len)
print('tar문장 최대 길이', max_tar_len)

src문장 최대 길이 22
tar문장 최대 길이 74


In [ ]:
# 패딩하기전에 확인해보기
print(encoder_input[0])
print(decoder_input[0])
print(decoder_target[0])

[30, 64, 10]
[1, 46, 50, 3, 4, 2]
[46, 50, 3, 4, 2]


In [ ]:
# 일전의 imdb 데이터 예제에선  padding='pre' 로 했었다.
# '긍정/부정' 분류 판정에 있어서, 문장의 중요한 정보가 시퀀스 '뒤쪽'에 있을가능성이 컸기 때문이다.

# 반면에 이번에는 번역이다.  차례대로 '처음'부터 순차적으로 번역결과 출력이 되어야 한다
# 그래서 padding='post' 로 합니다

# 기본적인 패딩값은 0 으로 채워진다   (그래서 사전의 인덱스 0 를 패딩을 위해 비워둔거다)


In [ ]:
# ★ 한번만 실행하기!
encoder_input_padded = pad_sequences(encoder_input, maxlen=max_src_len, padding='post')
decoder_input_padded = pad_sequences(decoder_input, maxlen=max_tar_len, padding='post')
decoder_target_padded = pad_sequences(decoder_target, maxlen=max_tar_len, padding='post')

print(encoder_input_padded.shape)
print(decoder_input_padded.shape)
print(decoder_target_padded.shape)


(60000, 22)
(60000, 74)
(60000, 74)


In [ ]:
#
print(lines.src[0])
print(encoder_input[0])
print(encoder_input_padded[0])

Go.
[30, 64, 10]
[30 64 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]


## One-hot encoding

In [ ]:
from tensorflow.keras.utils import to_categorical

In [ ]:
# num_classes= 값이 주어지지 않아도 '사전의 크기' 만큼 0,1 로 채워진 결과가 나올거다
encoder_input_oh = to_categorical(encoder_input_padded)
decoder_input_oh = to_categorical(decoder_input_padded)
decoder_target_oh = to_categorical(decoder_target_padded)


In [ ]:
# '사전의 크기' 만큼 0 과 1 로 채워진 one-hot vector 가 만들어질거다
# 확인
print(src_vocab_size, tar_vocab_size)

print(encoder_input_oh.shape)
print(decoder_input_oh.shape)
print(decoder_target_oh.shape)


80 102
(60000, 22, 80)
(60000, 74, 102)
(60000, 74, 102)


# 모델 seq2seq

![](https://wikidocs.net/images/page/24996/%EC%9D%B8%EC%BD%94%EB%8D%94%EB%94%94%EC%BD%94%EB%8D%94%EB%AA%A8%EB%8D%B8.PNG)

## 인코더(Encoder) 구성



* encoder는 입력 문장을 받는 여러 개의 RNN cell
* 입력은 단어 토큰화로 단어 단위로 쪼개지고, 이는 각 시점의 encoder 입력이 됌
* encoder는 모든 단어를 입력받고 마지막 시점의 은닉 상태를 decoder RNN cell의 첫번째 은닉 상태로 넘겨주며,
이를 컨텍스트 벡터(context vector)라고 함
* encoder는! **입력 시퀀스**를 → **컨텍스트 벡터**라는 고정 길이 벡터로 압축해야 함 (이 컨텍스트 벡터가 decoder) 로 넘어가게 됨




*   encoder의 구성은 일반 lstm 모델과 동일하나..
*   lstm 안의 return_state=True 로 설정하여 은닉 상태를 반환해줘야 한다! seq2seq 모델을 구성할 때 필요!




In [ ]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model


In [ ]:
encoder_inputs = Input(shape=(None, src_vocab_size)) # 글자사전의 크기만큼의 원핫벡터가 입력
encoder_lstm = LSTM(units=256, return_state=True) # 은닉상태 출력을 위해 return_state=True

# encoder_outputs은 여기서는 불필요
# hidden state(은닉상태) 와 cell state(셀상태) 리턴됨 (return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)

# LSTM은 바닐라 RNN과는 달리 'state'가 두개. '은닉 상태'와 '셀 상태'.
#  => 이들을 묶어서 context vector 형성
encoder_states = [state_h, state_c]
# 나중에 위 값을 디코더 전달해줄거다


In [ ]:
print('encoder_inputs:', encoder_inputs.shape)
print('encoder_outputs:', encoder_outputs.shape)
print('state_h:', state_h.shape)
print('state_c:', state_c.shape)


encoder_inputs: (None, None, 80)
encoder_outputs: (None, 256)
state_h: (None, 256)
state_c: (None, 256)
